In [1]:
# -*- coding: utf-8 -*-
"""Graphe Supervise.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/15it1QhJaYpP82YDQx57IXuY1UZYc0cdf
"""

!wget https://huggingface.co/datasets/Jgmorenof/teaching_tools_2025/resolve/main/chef-douvre.zip

!unzip chef-douvre.zip

# 1. INSTALLATION ET IMPORTS
!pip install -q lxml pandas numpy scikit-learn sentence-transformers lightgbm networkx tqdm

import os, re, glob
import numpy as np
import pandas as pd
from lxml import etree
from collections import Counter
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             homogeneity_score, completeness_score, v_measure_score)
from lightgbm import LGBMClassifier
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import warnings

warnings.filterwarnings('ignore')
print(" Imports terminés. Environnement prêt.")

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
   creating: chef-douvre/venv/lib/python3.12/site-packages/setuptools/tests/config/downloads/
  inflating: chef-douvre/venv/lib/python3.12/site-packages/setuptools/tests/config/downloads/__init__.py  
   creating: chef-douvre/venv/lib/python3.12/site-packages/setuptools/tests/config/downloads/__pycache__/
  inflating: chef-douvre/venv/lib/python3.12/site-packages/setuptools/tests/config/downloads/__pycache__/__init__.cpython-312.pyc  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/setuptools/tests/config/downloads/__pycache__/preload.cpython-312.pyc  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/setuptools/tests/config/downloads/preload.py  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/setuptools/tests/config/setupcfg_examples.txt  
  inflating: chef-douvre/venv/lib/python3.12/site-packages/setuptools/tests/config/test_pyprojecttoml.py  
  inflating: chef-douvre/venv/li

In [2]:
# 2. CONFIGURATION ET PARAMÈTRES
XML_FOLDER  = "/content/chef-douvre/AS_TrainingSet_BnF_NewsEye_v2"
THRESHOLD   = 0.85    # Seuil pour validation finale dans le graphe
K_NEIGHBORS = 20      # Nombre de voisins KNN
N_FILES     = 183

LGBM_PARAMS = dict(
    n_estimators      = 500,
    learning_rate     = 0.04,
    num_leaves        = 63,
    min_child_samples = 20,
    class_weight      = 'balanced',
    random_state      = 42,
    verbose           = -1,
)

FEAT_NAMES =[
    'cos_sim',
    'abs_dcx', 'abs_dcy', 'dcx', 'dcy', 'dist_eucl',
    'col_gap', 'same_col', 'adj_col',
    'w_i', 'w_j', 'h_i', 'h_j',
    'abs_dw', 'abs_dh', 'w_ratio', 'h_ratio',
    'hw_ratio_i', 'hw_ratio_j',
    'same_col_bool', 'read_order', 'j_below', 'j_right',
    'area_i', 'area_j', 'area_ratio',
    'n_lines_i', 'n_lines_j', 'n_lines_ratio',
    'cy_mean',
]

print(" Paramètres configurés.")

 Paramètres configurés.


In [3]:
# 3. PARSING DU XML
def detect_ns(root):
    tag = root.tag
    if tag.startswith('{'): return {'ns': tag[1:tag.index('}')]}
    return {}

def find_any(el, tag, ns):
    return el.find(f".//ns:{tag}", ns) if ns else el.find(f".//{tag}")

def findall_any(el, tag, ns):
    return el.findall(f'.//ns:{tag}', ns) if ns else el.findall(f'.//{tag}')

def findtext_any(el, tag, ns, default=""):
    e = find_any(el, tag, ns)
    return e.text.strip() if e is not None and e.text else default

def get_coords(pts_str):
    pts =[list(map(int, p.split(','))) for p in pts_str.split()]
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    return min(xs), min(ys), max(xs)-min(xs), max(ys)-min(ys)

def extract_gt_id(custom_attr):
    m = re.search(r'structure\s*{\s*id:\s*(a\d+)', custom_attr)
    return m.group(1).strip() if m else 'NO_ID'

def load_data(limit=N_FILES):
    xml_files = sorted(glob.glob(os.path.join(XML_FOLDER, "*.xml")))[:limit]
    data, skipped =[], 0
    print(f" Parcours de {len(xml_files)} fichiers...")

    for path in tqdm(xml_files, desc="Parsing XML"):
        try:
            tree  = etree.parse(path); root = tree.getroot()
            NS    = detect_ns(root)
            page  = find_any(root, 'Page', NS)
            if page is None: skipped += 1; continue

            p_w   = float(page.get("imageWidth",  1))
            p_h   = float(page.get("imageHeight", 1))
            fname = os.path.basename(path)

            for r in findall_any(root, 'TextRegion', NS):
                lines = findall_any(r, 'TextLine', NS)
                text  = " ".join([findtext_any(l,'Unicode',NS) for l in lines]).strip()
                if not text: continue
                n_lines = len(lines)

                ids     =[extract_gt_id(l.get('custom','')) for l in lines]
                true_id = Counter([i for i in ids if i!='NO_ID']).most_common(1)[0][0] \
                          if any(i!='NO_ID' for i in ids) else 'NO_ID'
                ce = find_any(r, 'Coords', NS)

                if ce is None: continue
                pts_str = ce.get('points','')
                if not pts_str: continue

                x, y, w, h = get_coords(pts_str)
                data.append({
                    'filename':          fname,
                    'id_region':         r.get('id'),
                    'text':              text,
                    'n_lines':           n_lines,
                    'article_id':        true_id,
                    'article_id_global': f"{fname}__{true_id}" if true_id!='NO_ID' else 'NO_ID',
                    'cx': (x+w/2)/p_w,  'cy': (y+h/2)/p_h,
                    'w':  w/p_w,         'h':  h/p_h,
                })
        except Exception as e:
            print(f"[WARN] {os.path.basename(path)}: {e}"); skipped += 1

    if not data: raise ValueError("Le DataFrame est vide. Vérifiez XML_FOLDER.")
    df = pd.DataFrame(data)
    print(f"\n Terminé : {len(df)} blocs | {df['filename'].nunique()} fichiers valides | {skipped} ignorés.")
    return df

df = load_data()

 Parcours de 183 fichiers...


Parsing XML:   0%|          | 0/183 [00:00<?, ?it/s]


 Terminé : 62343 blocs | 183 fichiers valides | 0 ignorés.


In [4]:
# 4. EMBEDDINGS SÉMANTIQUES AVEC SBERT
print(" Initialisation du modèle SBERT (paraphrase-multilingual-mpnet-base-v2)...")
sbert = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2',truncate_dim=384)

def encode_texts(texts, bs=64):
    embs =[]
    for i in tqdm(range(0, len(texts), bs), desc="Encodage texte"):
        # Les embeddings paraphrase-multilingual-mpnet-base-v2 ont un norm naturel élevé, on s'assure de les normaliser
        embs.append(sbert.encode(texts[i:i+bs], normalize_embeddings=True))
    return np.vstack(embs)

text_emb = encode_texts(df['text'].tolist())
print(f" {text_emb.shape[0]} blocs de texte transformés en vecteurs (Dimensions: {text_emb.shape[1]}).")

 Initialisation du modèle SBERT (paraphrase-multilingual-mpnet-base-v2)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encodage texte:   0%|          | 0/975 [00:00<?, ?it/s]

 62343 blocs de texte transformés en vecteurs (Dimensions: 384).


In [5]:
# 5. FEATURE ENGINEERING : Calcul entre blocs
def det_col(cx, n=6): return min(int(cx*n), n-1)

def pair_features(ri, rj, ei, ej):
    cos        = float(cosine_similarity([ei],[ej])[0,0])
    dcx        = ri['cx'] - rj['cx']
    dcy        = ri['cy'] - rj['cy']
    ci, cj     = det_col(ri['cx']), det_col(rj['cx'])
    dist       = np.hypot(dcx, dcy)
    same_col   = int(abs(dcx) < 0.12)
    read_order = int(dcy > 0) if same_col else int(dcx > 0)
    area_i     = ri['w'] * ri['h']
    area_j     = rj['w'] * rj['h']
    nl_i, nl_j = float(ri.get('n_lines',1)), float(rj.get('n_lines',1))

    return np.array([
        cos,
        abs(dcx), abs(dcy), dcx, dcy, dist,
        abs(ci-cj), float(ci==cj), float(abs(ci-cj)==1),
        ri['w'], rj['w'], ri['h'], rj['h'],
        abs(ri['w']-rj['w']), abs(ri['h']-rj['h']),
        ri['w'] / max(rj['w'], 1e-6),
        ri['h'] / max(rj['h'], 1e-6),
        ri['h'] / max(ri['w'], 1e-6),
        rj['h'] / max(rj['w'], 1e-6),
        same_col, read_order,
        float(rj['cy'] > ri['cy']),
        float(rj['cx'] > ri['cx']),
        area_i, area_j, area_i / max(area_j, 1e-9),
        nl_i, nl_j, nl_i / max(nl_j, 1e-6),
        (ri['cy'] + rj['cy']) / 2,
    ])

print(" Construction du réseau par paires (KNN-Based)...")
page_data = {}

for fname, dfp in tqdm(df.groupby('filename'), desc="Couplage par pages"):
    idx    = dfp.index.tolist()
    emb_p  = text_emb[idx]
    dfp_r  = dfp.reset_index(drop=True)
    coords = dfp_r[['cx','cy']].values
    pairs, feats, labels = [], [],[]

    for ii in range(len(dfp_r)):
        # Trouver les voisins proches spatialement (incluant sauts potentiels)
        dists = np.linalg.norm(coords - coords[ii], axis=1)
        for jj in np.argsort(dists)[1:K_NEIGHBORS+1]:
            if ii >= jj: continue
            ri, rj = dfp_r.iloc[ii], dfp_r.iloc[jj]
            feat   = pair_features(ri, rj, emb_p[ii], emb_p[jj])

            # Ground Truth positive/négative
            same = int(ri['article_id'] == rj['article_id'] and ri['article_id'] != 'NO_ID')

            pairs.append((idx[ii], idx[jj])) # <-- Attention index Global
            feats.append(feat)
            labels.append(same)

    page_data[fname] = {
        'pairs':  pairs,
        'feats':  np.array(feats) if feats else np.empty((0, len(FEAT_NAMES))),
        'labels': np.array(labels),
    }

total = sum(len(v['pairs']) for v in page_data.values())
pos   = sum(v['labels'].sum() for v in page_data.values())
print(f" {total} paires calculées | Cible Positives: {int(pos)} ({100*pos/total:.1f}%)")

 Construction du réseau par paires (KNN-Based)...


Couplage par pages:   0%|          | 0/183 [00:00<?, ?it/s]

 632965 paires calculées | Cible Positives: 356922 (56.4%)


In [6]:
# 6. ENTRAÎNEMENT ET PRÉDICTION
print(f" Préparation de l'évaluation avec Split Train/Test pur…")

# On récupère toutes les pages traitées
all_pages = list(page_data.keys())

# 80% d'entraînement, 20% pour le test final.
train_pages, test_pages = train_test_split(all_pages, test_size=0.20, random_state=42)

print(f" Pages pour l'entraînement : {len(train_pages)}")
print(f" Pages pour le Test pur    : {len(test_pages)}")

# A) CRÉATION DU TRAINING SET
print("\n Entraînement de LightGBM en cours...")
X_tr = np.vstack([page_data[p]['feats'] for p in train_pages if len(page_data[p]['feats']) > 0])
y_tr = np.concatenate([page_data[p]['labels'] for p in train_pages])

clf = LGBMClassifier(**LGBM_PARAMS)
clf.fit(X_tr, y_tr)
print(" Modèle entraîné avec succès.")

# B) PRÉDICTION & EVALUATION SUR LE TEST SET
print(f"\n Evaluation sur le Test Set (Seuil Graphe: {THRESHOLD})…")
print(f"  {'Fichier Evalué':40s} | Blocs | Réel_Art | Préd_Art | Score ARI")
print("  " + "─"*72)

all_pred_labels = {}
ari_pages       =[]

for test_page in test_pages:

    # 1. On récupère la page vierge de tout entraînement
    pd_test = page_data[test_page]
    if len(pd_test['feats']) == 0: continue

    # 2. Prédiction LightGBM
    probs = clf.predict_proba(pd_test['feats'])[:,1]

    # 3. Graphe et Coupures
    dfp      = df[df['filename'] == test_page]
    idx_page = set(dfp.index.tolist())
    G        = nx.Graph()
    G.add_nodes_from(idx_page)

    for (idx_pair, (i, j)), p in zip(enumerate(pd_test['pairs']), probs):
        if p > THRESHOLD:

            ri = df.loc[i]
            rj = df.loc[j]
            dy = rj['cy'] - ri['cy']

            # ANTI-SOUS-SEGMENTATION
            if dy > 0.15 and p < 0.95:
                continue

            if rj['n_lines'] < 2 and p < 0.90:
                continue

            G.add_edge(i, j, weight=float(p))

    # 4. Affectation Clusters
    pred = {}
    for cid, comp in enumerate(nx.connected_components(G)):
        for node in comp:
            pred[node]            = cid
            all_pred_labels[node] = f"{test_page}_c{cid}"

    # 5. Scores
    yt     = dfp['article_id'].astype(str).values
    yp     = np.array([str(pred.get(i, -1)) for i in dfp.index])
    n_real = dfp['article_id'].nunique()
    n_pred = len(set(pred.values()))

    ari_f  = adjusted_rand_score(yt, yp) if len(set(yt)) >= 2 else float('nan')
    if not np.isnan(ari_f): ari_pages.append(ari_f)

    print(f"  {test_page:40s} | {len(dfp):5d} | {n_real:8d} | {n_pred:8d} | {ari_f:.3f}")

print("\n Prédictions Test Terminées !")

 Préparation de l'évaluation avec Split Train/Test pur…
 Pages pour l'entraînement : 146
 Pages pour le Test pur    : 37

 Entraînement de LightGBM en cours...
 Modèle entraîné avec succès.

 Evaluation sur le Test Set (Seuil Graphe: 0.85)…
  Fichier Evalué                           | Blocs | Réel_Art | Préd_Art | Score ARI
  ────────────────────────────────────────────────────────────────────────
  18720715_1-0004.xml                      |   503 |       45 |       71 | 0.351
  18880115_1-0003.xml                      |   293 |       53 |       80 | 0.266
  19240115_1-0001.xml                      |   201 |       16 |       50 | 0.449
  19100115_1-0003.xml                      |   287 |       35 |       57 | 0.648
  19200115_1-0003.xml                      |   317 |       64 |       93 | 0.575
  18710715_1-0004.xml                      |   435 |       50 |       56 | 0.970
  18740715_1-0001.xml                      |   197 |       18 |       54 | 0.395
  18980715_1-0002.xml           

In [7]:
# 7. METRIQUES FINALES SUR LE SET DE TEST

# 1. On applique les prédictions au DataFrame.
df['predicted_cluster'] =[all_pred_labels.get(i, "TRAIN_SET_IGNORE") for i in df.index]

# 2. On Isole EXCLUSIVEMENT les pages de Test pour le calcul des scores
df_test = df[df['filename'].isin(test_pages)]

# 3. On enlève le bruit ('NO_ID' Ground truth)
mask_g = df_test['article_id_global'] != 'NO_ID'
df_g   = df_test[mask_g]

yt_g   = df_g['article_id_global'].astype(str).values
yp_g   = df_g['predicted_cluster'].astype(str).values

# 4. Statistiques Globales
ari_g  = adjusted_rand_score(yt_g, yp_g)
nmi_g  = normalized_mutual_info_score(yt_g, yp_g)
hom_g  = homogeneity_score(yt_g, yp_g)
comp_g = completeness_score(yt_g, yp_g)
vm_g   = v_measure_score(yt_g, yp_g)

print("\n" + "═"*58)
print("   RÉSULTATS DE SEGMENTATION GLOBAUX SUR TEST ")
print("═"*58)
print(f"   ARI Moyen/Page   : {np.mean(ari_pages):.4f} ± {np.std(ari_pages):.3f} ")
print(f"  •  ARI Base Global  : {ari_g:.4f}")
print(f"  •  NMI Global       : {nmi_g:.4f}")
print(f"  •  V-Mesure Globale : {vm_g:.4f}")
print("═"*58)


══════════════════════════════════════════════════════════
   RÉSULTATS DE SEGMENTATION GLOBAUX SUR TEST 
══════════════════════════════════════════════════════════
   ARI Moyen/Page   : 0.5089 ± 0.190 
  •  ARI Base Global  : 0.7429
  •  NMI Global       : 0.9206
  •  V-Mesure Globale : 0.9206
══════════════════════════════════════════════════════════


In [8]:
# 8. EXPORTATION DES PRÉDICTIONS
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    output_file = "predictions_finales.csv"

    export_df = df[['filename', 'id_region', 'text', 'article_id', 'article_id_global', 'predicted_cluster']]
    export_df.to_csv(output_file, index=False)

print(f"{output_file}")

predictions_finales.csv


In [9]:
# 9. VISUALISATION DES RESULTATS
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image


# CHOIX DE LA PAGE À VISUALISER
TARGET_FILE = "18970715_1-0001.xml"

IMG_FOLDER  = XML_FOLDER

def visualize_prediction(dataframe, filename, img_folder):

    # 1. Extraire les blocs liés à notre page
    df_page = dataframe[dataframe['filename'] == filename].copy()
    if df_page.empty:
        print(f" Erreur: Le fichier '{filename}' est introuvable.")
        return

    # 2. Ouverture de l'Image
    path_img = os.path.join(img_folder, filename.replace(".xml", ".jpg"))
    if not os.path.exists(path_img):
        print(f" Erreur: L'image jpg est introuvable ici : {path_img}")
        return

    img = Image.open(path_img)
    img_w, img_h = img.size

    # 3. Calculer localement l'ARI
    mask    = df_page['article_id'] != 'NO_ID'
    df_eval = df_page[mask]

    if len(df_eval) > 1:
        ari = adjusted_rand_score(df_eval['article_id'], df_eval['predicted_cluster'])
        vm  = v_measure_score(df_eval['article_id'], df_eval['predicted_cluster'])
    else:
        ari, vm = 0, 0

    print(f"\n─────────────🔎  ANALYSE DE : {filename} ───────────────────")
    print(f"   Blocs reconnus : {len(df_page)}")
    print(f"   Vrais Articles : {df_eval['article_id'].nunique()}")
    print(f"   Vos Clusters   : {df_eval['predicted_cluster'].nunique()}")
    print(f"    SCORE ARI       : {ari:.4f}")
    print(f"    SCORE V-Measure : {vm:.4f}\n")

    # 4. PRÉPARATION DU TRACÉ
    fig, ax = plt.subplots(1, 2, figsize=(26, 18))
    cmap = plt.get_cmap('tab20')

    # GAUCHE (VÉRITÉ TERRAIN)
    ax[0].imshow(img)
    ax[0].set_title(" Vérité Terrain", fontsize=22, fontweight='bold', pad=20, color='darkgreen')
    ax[0].axis('off')

    true_ids = df_page['article_id'].unique().tolist()

    for _, row in df_page.iterrows():
        if row['article_id'] == 'NO_ID': continue

        color = cmap(true_ids.index(row['article_id']) % 20)

        x_ratio = row['cx'] - (row['w'] / 2)
        y_ratio = row['cy'] - (row['h'] / 2)

        rx, ry = x_ratio * img_w, y_ratio * img_h
        rw, rh = row['w'] * img_w, row['h'] * img_h

        rect = patches.Rectangle((rx, ry), rw, rh, linewidth=3, edgecolor=color, facecolor=color, alpha=0.35)
        ax[0].add_patch(rect)

    # DROITE (MODÈLE PRÉDIT)
    ax[1].imshow(img)
    ax[1].set_title(f" Prédictions paraphrase-multilingual-mpnet-base-v2 ", fontsize=22, fontweight='bold', pad=20, color='darkred')
    ax[1].axis('off')

    pred_ids = df_page['predicted_cluster'].unique().tolist()

    for _, row in df_page.iterrows():
        if row['article_id'] == 'NO_ID': continue

        color = cmap(pred_ids.index(row['predicted_cluster']) % 20)

        x_ratio = row['cx'] - (row['w'] / 2)
        y_ratio = row['cy'] - (row['h'] / 2)

        rx, ry = x_ratio * img_w, y_ratio * img_h
        rw, rh = row['w'] * img_w, row['h'] * img_h

        rect = patches.Rectangle((rx, ry), rw, rh, linewidth=3, edgecolor=color, facecolor=color, alpha=0.35)
        ax[1].add_patch(rect)

    plt.tight_layout()
    plt.show()

# Affichage visuel :
visualize_prediction(df, TARGET_FILE, IMG_FOLDER)

# CHOIX DE LA PAGE À VISUALISER
TARGET_FILE = "18970715_1-0001.xml"

def visualiser_df(dataframe, filename, img_folder):
    # 1. Isoler les données en un clin d'œil depuis le DataFrame
    df_page = dataframe[dataframe['filename'] == filename].copy()
    if df_page.empty: return print(f" Erreur: Fichier '{filename}' non trouvé dans df.")

    # 2. Ouverture de l'Image
    path_img = os.path.join(img_folder, filename.replace(".xml", ".jpg"))
    if not os.path.exists(path_img): return print(f" Image jpg introuvable : {path_img}")
    img = Image.open(path_img)
    img_w, img_h = img.size

    # On ignore le bruit ("NO_ID" ou autre) pour comptage propre
    df_eval = df_page[df_page['article_id'] != 'NO_ID']

    fig, ax = plt.subplots(1, 2, figsize=(24, 18))
    cmap = plt.get_cmap('tab20')

    # IDs uniques pour figer les couleurs GT / PRED
    unique_gt_ids   = df_eval['article_id'].unique().tolist()
    unique_pred_ids = df_eval['predicted_cluster'].unique().tolist()

    # IMAGE GAUCHE (VÉRITÉ TERRAIN)
    ax[0].imshow(img, alpha=0.6)
    ax[0].set_title(f"VRAIE STRUCTURE ({len(unique_gt_ids)} articles réels)", fontsize=18)
    ax[0].axis('off')

    for _, row in df_eval.iterrows():
        c = cmap(unique_gt_ids.index(row['article_id']) % 20)
        w, h = row['w'] * img_w, row['h'] * img_h
        x, y = (row['cx'] * img_w) - (w/2), (row['cy'] * img_h) - (h/2)

        rect = patches.Rectangle((x, y), w, h, linewidth=1, edgecolor='black', facecolor=(c[0],c[1],c[2], 0.5))
        ax[0].add_patch(rect)

    # IMAGE DROITE (MODÈLE PRÉDIT)
    ax[1].imshow(img, alpha=0.6)
    ax[1].set_title(f"PRÉDICTION ({len(unique_pred_ids)} clusters paraphrase-multilingual-mpnet-base-v2)", fontsize=18)
    ax[1].axis('off')

    for _, row in df_eval.iterrows():
        if row['predicted_cluster'] == "TRAIN_SET_IGNORE": continue

        c = cmap(unique_pred_ids.index(row['predicted_cluster']) % 20)

        w, h = row['w'] * img_w, row['h'] * img_h
        x, y = (row['cx'] * img_w) - (w/2), (row['cy'] * img_h) - (h/2)

        rect = patches.Rectangle((x, y), w, h, linewidth=1, edgecolor='black', facecolor=(c[0],c[1],c[2], 0.5))
        ax[1].add_patch(rect)

    plt.tight_layout()
    plt.show()

visualiser_df(df, TARGET_FILE, XML_FOLDER)

Output hidden; open in https://colab.research.google.com to view.